In [15]:
import data_preprocessing as dp

In [16]:
# 测试一下
tensor_input, original_img = dp.preprocess_image("./data/tree.jpg")
print(tensor_input.shape) # 输出应该是 torch.Size([1, 3, 640, 640])

print("Converted bounding boxes: " + str(dp.xywh_to_xyxy(tensor_input)))

torch.Size([1, 3, 640, 640])
Converted bounding boxes: tensor([[[[0.2098, 0.2039, 0.6373,  ..., 0.8078, 0.8000, 0.8196],
          [0.2216, 0.2176, 0.6412,  ..., 0.8118, 0.8157, 0.8157],
          [0.2235, 0.2098, 0.6471,  ..., 0.8431, 0.8196, 0.8235],
          ...,
          [0.2412, 0.2510, 0.6608,  ..., 0.4157, 0.4118, 0.4157],
          [0.2412, 0.2314, 0.6765,  ..., 0.4196, 0.4353, 0.4078],
          [0.2294, 0.1882, 0.6569,  ..., 0.3961, 0.3765, 0.3843]],

         [[0.3137, 0.3137, 0.9490,  ..., 0.8588, 0.8627, 0.8667],
          [0.3275, 0.3275, 0.9588,  ..., 0.8627, 0.8667, 0.8706],
          [0.3255, 0.3118, 0.9608,  ..., 0.8902, 0.8745, 0.8902],
          ...,
          [0.2157, 0.2157, 0.6235,  ..., 0.4235, 0.4196, 0.4275],
          [0.2059, 0.1784, 0.6333,  ..., 0.4235, 0.4314, 0.4157],
          [0.2098, 0.1647, 0.6294,  ..., 0.4118, 0.4078, 0.4078]],

         [[0.4471, 0.4510, 1.3490,  ..., 0.9216, 0.9137, 0.9176],
          [0.4608, 0.4588, 1.3745,  ..., 0.9255, 0.92

In [17]:
import cv2
import numpy as np

def visualize_bbox(img_path, yolo_labels, class_names):
    """
    读取一张图片，并根据 YOLO 格式的标签在图上画出边界框和类别
    yolo_labels 格式: [[class_id, x_center, y_center, w, h], ...] (皆为 0~1 的相对坐标)
    """
    img = cv2.imread(img_path)
    h, w, _ = img.shape  # 获取原图的绝对像素宽高
    
    for label in yolo_labels:
        cls_id, x_c, y_c, bbox_w, bbox_h = label
        
        # 1. 核心转换：将 0~1 的相对坐标 还原为 原图的绝对像素坐标
        pixel_x_c = x_c * w
        pixel_y_c = y_c * h
        pixel_w = bbox_w * w
        pixel_h = bbox_h * h
        
        # 2. 从 [中心点, 宽高] 转换为 OpenCV 绘图需要的 [左上角, 右下角]
        xmin = int(pixel_x_c - pixel_w / 2)
        ymin = int(pixel_y_c - pixel_h / 2)
        xmax = int(pixel_x_c + pixel_w / 2)
        ymax = int(pixel_y_c + pixel_h / 2)
        
        # 3. 使用 OpenCV 画框 (注意：cv2.rectangle 接收的是像素整型坐标)
        # 颜色用 BGR 格式表示，这里画一根绿色的框，粗细为 2 像素
        cv2.rectangle(img, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
        
        # 4. 把类别标签字样写在框的上方
        text = f"{class_names[int(cls_id)]}"
        cv2.putText(img, text, (xmin, ymin - 5), cv2.FONT_HERSHEY_SIMPLEX, 
                    0.6, (0, 255, 0), 2, cv2.LINE_AA)
        
    return img

def convert_to_yolo_label(img_shape, xmin, ymin, xmax, ymax, class_id=0):
    """
    逆向思考：将你在 OpenCV 里肉眼看到的像素坐标，转换为 YOLO 训练所需的归一化文本数据
    img_shape: (H, W)
    """
    h, w = img_shape
    
    # 1. 计算像素层面的中心点和宽高
    pixel_cx = (xmin + xmax) / 2
    pixel_cy = (ymin + ymax) / 2
    pixel_w = xmax - xmin
    pixel_h = ymax - ymin
    
    # 2. 归一化：除以原图的宽和高，使其落在 0~1 之间
    x_center = pixel_cx / w
    y_center = pixel_cy / h
    width = pixel_w / w
    height = pixel_h / h
    
    # 返回符合 YOLO 标准格式的字符串
    return f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"

In [18]:
# 假设我们有两个目标，类别分别是 0: "tree", 1: "cloud"
classes = ["tree", "cloud"]

# 模拟网络预测出来的或者我们自己手写的 YOLO 相对坐标数据
test_labels = [
    # [cls_id, x_center, y_center, w, h]
    [0, 0.3, 0.4, 0.2, 0.3],  # 一个位于左上方区域的目标
    [1, 0.7, 0.6, 0.25, 0.4]  # 一个位于右下方区域的目标
]

# 渲染图片
result_img = visualize_bbox("./data/tree.jpg", test_labels, classes)

# 保存查看结果
cv2.imwrite("annotated_result.jpg", result_img)

True

In [19]:
import torch
import torch.nn as nn

class MiniBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        # 假设输入是 [B, 3, 640, 640]
        # P1 下采样: 640 -> 320
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1)
        # P2 下采样: 320 -> 160
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1)
        # P3 下采样: 160 -> 80 (YOLOv5 的浅层特征图，负责检测小物体)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        return x

# 1. 实例化我们的极简网络
model = MiniBackbone()

# 2. 模拟一个 Batch 大小为 1，尺寸为 640x640 的 RGB 图像张量
dummy_input = torch.randn(1, 3, 640, 640)

# 3. 前向传播
feature_map = model(dummy_input)
print(f"最终特征图的维度: {feature_map.shape}")
# 输出应该是: torch.Size([1, 64, 80, 80])
# 这里的 80x80 就是网格(Grid)！整张图被切成了 80x80 = 6400 个小格子。

最终特征图的维度: torch.Size([1, 64, 80, 80])


In [20]:
# 原图上的边界框中心点像素坐标
pixel_x = 200.0
pixel_y = 400.0

# 计算下采样倍数 (Stride)。原图 640 变为了 80，缩放了 640 / 80 = 8 倍
stride = 640 / feature_map.shape[2] # 8.0

# 计算在 80x80 网格中的浮点坐标
grid_x = pixel_x / stride  # 200 / 8 = 25.0
grid_y = pixel_y / stride  # 400 / 8 = 50.0

# 取整，得到它所属的特定网格行与列的索引 (Grid Index)
grid_col = int(grid_x) # 第 25 列
grid_row = int(grid_y) # 第 50 行

print(f"该目标在原图上的坐标为 ({pixel_x}, {pixel_y})")
print(f"在 80x80 的特征图中，它落在了第 {grid_row} 行，第 {grid_col} 列的格子里！")

该目标在原图上的坐标为 (200.0, 400.0)
在 80x80 的特征图中，它落在了第 50 行，第 25 列的格子里！


In [21]:
import torch
import math

# =====================================================================
# 1. 初始化数据：定义原图尺寸以及两个物体的【像素真实坐标】
# =====================================================================
IMG_SIZE = 640  # 输入图片的宽高都是 640

# 目标 A: 小乒乓球 -> 中心点 (160, 160), 宽高 (32, 32)
ball_box = {"name": "乒乓球", "cx": 160.0, "cy": 160.0, "w": 32.0, "h": 32.0}

# 目标 B: 大卡车 -> 中心点 (480, 400), 宽高 (320, 256)
truck_box = {"name": "大卡车", "cx": 480.0, "cy": 400.0, "w": 320.0, "h": 256.0}

targets = [ball_box, truck_box]

# =====================================================================
# 2. 定义 YOLO 经典的三层检测特征图 (P3, P4, P5)
# =====================================================================
# strides: 下采样倍数。原图尺寸除以 stride 就是特征图的网格大小
detect_strides = {
    "P3 (浅层/大尺寸)": 8,   # 640 / 8 = 80x80 网格
    "P4 (中层/中尺寸)": 16,  # 640 / 16 = 40x40 网格
    "P5 (深层/小尺寸)": 32   # 640 / 32 = 20x20 网格
}

print(f"--- 🚀 开始进行多尺度空间映射分析 ---")

# =====================================================================
# 3. 核心映射计算
# =====================================================================
for target in targets:
    print(f"\n📦 物体: 【{target['name']}】")
    print(f"   原图绝对像素中心点: (X={target['cx']}, Y={target['cy']}), 宽高: ({target['w']}x{target['h']})")
    
    # 模拟 YOLO 的多尺度选择逻辑（这里用简化版的尺寸契合度来演示）
    # 现实中 YOLOv5 靠 Anchor 比例分配，YOLOv8 靠 TAL 动态分配。
    # 我们根据物体的绝对大小，手动将它们分流到最适合的层：
    if target['w'] < 64:
        assigned_layer = "P3 (浅层/大尺寸)"
    elif target['w'] < 192:
        assigned_layer = "P4 (中层/中尺寸)"
    else:
        assigned_layer = "P5 (深层/小尺寸)"
        
    stride = detect_strides[assigned_layer]
    grid_size = IMG_SIZE // stride
    
    # 【最核心的数学映射】: 像素坐标 / 下采样倍数 = 特征图网格坐标
    grid_x = target['cx'] / stride
    grid_y = target['cy'] / stride
    
    # 取整得到矩阵的行与列索引 (注意：y对应Row行，x对应Col列)
    grid_col = math.floor(grid_x)
    grid_row = math.floor(grid_y)
    
    print(f"   🎯 分配层: {assigned_layer} (特征图网格尺寸为 {grid_size}x{grid_size})")
    print(f"   📍 映射到特征图的浮点坐标: (grid_x={grid_x:.2f}, grid_y={grid_y:.2f})")
    print(f"   🔥 最终负责预测该物体的网格索引 -> 【第 {grid_row} 行，第 {grid_col} 列】")

--- 🚀 开始进行多尺度空间映射分析 ---

📦 物体: 【乒乓球】
   原图绝对像素中心点: (X=160.0, Y=160.0), 宽高: (32.0x32.0)
   🎯 分配层: P3 (浅层/大尺寸) (特征图网格尺寸为 80x80)
   📍 映射到特征图的浮点坐标: (grid_x=20.00, grid_y=20.00)
   🔥 最终负责预测该物体的网格索引 -> 【第 20 行，第 20 列】

📦 物体: 【大卡车】
   原图绝对像素中心点: (X=480.0, Y=400.0), 宽高: (320.0x256.0)
   🎯 分配层: P5 (深层/小尺寸) (特征图网格尺寸为 20x20)
   📍 映射到特征图的浮点坐标: (grid_x=15.00, grid_y=12.50)
   🔥 最终负责预测该物体的网格索引 -> 【第 12 行，第 15 列】
